## Ingest Dimension Data into Bronze Layer

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, FloatType
import pyspark.sql.functions as F
from datetime import datetime
import os

In [0]:
catalog_name = 'ecommerce'

#### Setting up current date variable to Archive files once processed


In [0]:

archive_date = datetime.now().strftime("%Y-%m-%d")
print("archive_date : ",archive_date)

# If the pipeline may run multiple times per day, use a timestamp instead of only the date
# This prevents accidental overwrites when the same file name is processed multiple times on the same day.

#archive_date = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

archive_date :  2026-06-20


### Brands

In [0]:
# Define schema for the data file
brand_schema = StructType([
    StructField("brand_code", StringType(), False),
    StructField("brand_name", StringType(), True),
    StructField("category_code", StringType(), True),
])

#### - Metadata columns for Auditing -
**_ingested_at      :**	Timestamp when record entered Bronze

**_source_file      :**	Source CSV file name/path

**_load_date        :**    Partitioning and auditing

**_rescued_data     :**    (Optional Auto Loader column for bad records)

**_metadata.file_path:** If Using Databricks Auto Loader, then you can use this

In [0]:
raw_data_path = "/Volumes/ecommerce/source_data/raw/brands/*.csv"

df = spark.read.option('header', "true").option("delimeter", ",").schema(brand_schema).csv(raw_data_path)

# add metadata columns
df = df.withColumn("_source_file", F.col("_metadata.file_path")) \
       .withColumn("ingested_at", F.current_timestamp())

display(df.limit(5))       

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8444991654744571>, line 9
      5 # add metadata columns
      6 df = df.withColumn("_source_file", F.col("_metadata.file_path")) \
      7        .withColumn("ingested_at", F.current_timestamp())
----> 9 display(df.limit(5))

File /databricks/python_shell/lib/dbruntime/display.py:133, in Display.display(self, input, *args, **kwargs)
    131     pass
    132 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 133     self.display_connect_table(input, **kwargs)
    134 elif isinstance(input, ConnectDataFrame):
    135     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:93, in Display.display_connect_table(self, df, **kwargs)
     88 except Exception as e:
     89     raise type(
     90         e
     91     )("IPython shell encountered an error or was missing data, pl

In [0]:
archive_base_path = f"/Volumes/ecommerce/source_data/archive/{archive_date}/brands/"
print("archive_base_path : ",archive_base_path)

try:
    # Write to Bronze Delta Table
    (
        # --> .mode("overwrite") : Deletes/replaces existing data and writes only the new data.For, 
        # --> .mode("append")    : Adds new records to the existing table/files. Existing data remains.
        df.write
              .format("delta")
              .mode("overwrite")
              .option("mergeSchema", "true")
              .saveAsTable(f"{catalog_name}.bronze.brz_brands")
    )

    # Create archive folder
    dbutils.fs.mkdirs(archive_base_path)

    # Get all processed source files
    source_files = (
        df.select("_source_file")
          .distinct()
          .collect()
    )
    print("source_files : ",source_files)
    # Archive each file
    for row in source_files:
        source_file = row["_source_file"]
        file_name = os.path.basename(source_file)
        print("file_name : ",file_name)

        destination_file = f"{archive_base_path}{file_name}"
        dbutils.fs.mv(source_file, destination_file)

        print(f"Archived: {source_file} -> {destination_file}")

    print("Delta load and file archival completed successfully.")

except Exception as e:
    print(f"Pipeline failed: {str(e)}")
    raise

archive_base_path :  /Volumes/ecommerce/source_data/archive/2026-06-20/brands/
source_files :  [Row(_source_file='dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv')]
file_name :  brands.csv
Archived: dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv -> /Volumes/ecommerce/source_data/archive/2026-06-20/brands/brands.csv
Delta load and file archival completed successfully.


### Category

In [0]:
category_schema = StructType([
    StructField("category_code", StringType(), False),
    StructField("category_name", StringType(), True)
])

# Load data using the schema defined
raw_data_path = "/Volumes/ecommerce/source_data/raw/category/*.csv"

df_raw = spark.read.option("header", "true").option("delimiter", ",").schema(category_schema).csv(raw_data_path)

# Add metadata columns
df_raw = df_raw.withColumn("_ingested_at", F.current_timestamp()) \
               .withColumn("_source_file", F.col("_metadata.file_path"))


# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_category)
archive_base_path = f"/Volumes/ecommerce/source_data/archive/{archive_date}/category/"
print("archive_base_path : ",archive_base_path)
try:
    # Write to Bronze Delta Table
    (
        df_raw.write
              .format("delta")
              .mode("overwrite")
              .option("mergeSchema", "true")
              .saveAsTable(f"{catalog_name}.bronze.brz_category")
    )

    # Create archive folder
    dbutils.fs.mkdirs(archive_base_path)

    # Get all processed source files
    source_files = (
        df_raw.select("_source_file")
              .distinct()
              .collect()
    )
    print("source_files : ",source_files)
    # Archive each file
    for row in source_files:
        source_file = row["_source_file"]
        file_name = os.path.basename(source_file)
        print("file_name : ",file_name)

        destination_file = f"{archive_base_path}{file_name}"
        dbutils.fs.mv(source_file, destination_file)

        print(f"Archived: {source_file} -> {destination_file}")

    print("Delta load and file archival completed successfully.")

except Exception as e:
    print(f"Pipeline failed: {str(e)}")
    raise             

archive_base_path :  /Volumes/ecommerce/source_data/archive/2026-06-20/category/
source_files :  [Row(_source_file='dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv')]
file_name :  category.csv
Archived: dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv -> /Volumes/ecommerce/source_data/archive/2026-06-20/category/category.csv
Delta load and file archival completed successfully.


### Products

In [0]:
products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("sku", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand_code", StringType(), True),
    StructField("color", StringType(), True),
    StructField("size", StringType(), True),
    StructField("material", StringType(), True),
    StructField("weight_grams", StringType(), True),  #datatype is string due to incoming data contain anamolies
    StructField("length_cm", StringType(), True),     #datatype is string due to incoming data contain anamolies
    StructField("width_cm", FloatType(), True),
    StructField("height_cm", FloatType(), True),
    StructField("rating_count", IntegerType(), True),
    StructField("file_name", StringType(), False),
    StructField("ingest_timestamp", TimestampType(), False)
])

# Load data using the schema defined
raw_data_path = "/Volumes/ecommerce/source_data/raw/products/*.csv"

df = spark.read.option("header", "true").option("delimiter", ",").schema(products_schema).csv(raw_data_path) \
    .withColumn("_source_file", F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_products)
archive_base_path = f"/Volumes/ecommerce/source_data/archive/{archive_date}/products/"
print("archive_base_path : ",archive_base_path)
try:
    # Write to Bronze Delta Table
    (
        df.write
          .format("delta")
          .mode("overwrite")
          .option("mergeSchema", "true")
          .saveAsTable(f"{catalog_name}.bronze.brz_products")
    )

    # Create archive folder
    dbutils.fs.mkdirs(archive_base_path)

    # Get all processed source files
    source_files = (
        df.select("_source_file")
          .distinct()
          .collect()
    )
    print("source_files : ",source_files)
    # Archive each file
    for row in source_files:
        source_file = row["_source_file"]
        file_name = os.path.basename(source_file)
        print("file_name : ",file_name)

        destination_file = f"{archive_base_path}{file_name}"
        dbutils.fs.mv(source_file, destination_file)

        print(f"Archived: {source_file} -> {destination_file}")

    print("Delta load and file archival completed successfully.")

except Exception as e:
    print(f"Pipeline failed: {str(e)}")
    raise  

archive_base_path :  /Volumes/ecommerce/source_data/archive/2026-06-20/products/
source_files :  [Row(_source_file='dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv')]
file_name :  products.csv
Archived: dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv -> /Volumes/ecommerce/source_data/archive/2026-06-20/products/products.csv
Delta load and file archival completed successfully.


### Customers

In [0]:
customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("phone", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("country", StringType(), True),
    StructField("state", StringType(), True)
])

# Load data using the schema defined
raw_data_path ="/Volumes/ecommerce/source_data/raw/customers/*.csv"

df_raw = spark.read.option("header", "true").option("delimiter", ",").schema(customers_schema).csv(raw_data_path) \
    .withColumn("_source_file", F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_customers) 
archive_base_path = f"/Volumes/ecommerce/source_data/archive/{archive_date}/customers/"
print("archive_base_path : ",archive_base_path)
try:
    # Write to Bronze Delta Table
    (
        df_raw.write
              .format("delta")
              .mode("overwrite")
              .option("mergeSchema", "true")
              .saveAsTable(f"{catalog_name}.bronze.brz_customers")
    )

    # Create archive folder
    dbutils.fs.mkdirs(archive_base_path)

    # Get all processed source files
    source_files = (
        df_raw.select("_source_file")
              .distinct()
              .collect()
    )
    print("source_files : ",source_files)
    # Archive each file
    for row in source_files:
        source_file = row["_source_file"]
        file_name = os.path.basename(source_file)
        print("file_name : ",file_name)

        destination_file = f"{archive_base_path}{file_name}"
        dbutils.fs.mv(source_file, destination_file)

        print(f"Archived: {source_file} -> {destination_file}")

    print("Delta load and file archival completed successfully.")

except Exception as e:
    print(f"Pipeline failed: {str(e)}")
    raise  

archive_base_path :  /Volumes/ecommerce/source_data/archive/2026-06-20/customers/
source_files :  [Row(_source_file='dbfs:/Volumes/ecommerce/source_data/raw/customers/customers.csv')]
file_name :  customers.csv
Archived: dbfs:/Volumes/ecommerce/source_data/raw/customers/customers.csv -> /Volumes/ecommerce/source_data/archive/2026-06-20/customers/customers.csv
Delta load and file archival completed successfully.


### Date

In [0]:

# Define schema for the data file
date_schema = StructType([
    StructField("date", StringType(), True),           # Raw date in string format
    StructField("year", IntegerType(), True),          # Year
    StructField("day_name", StringType(), True),       # Day name (can be mixed case)
    StructField("quarter", IntegerType(), True),       # Quarter
    StructField("week_of_year", IntegerType(), True),  # Week of year (can be negative)
])

# Load data using the schema defined
raw_data_path = f"/Volumes/ecommerce/source_data/raw/date/*.csv" 

df_raw = spark.read.option("header", "true").option("delimiter", ",").schema(date_schema).csv(raw_data_path)

# Add metadata columns
df_raw = df_raw.withColumn("_ingested_at", F.current_timestamp()) \
               .withColumn("_source_file", F.col("_metadata.file_path"))

# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_calendar) 
archive_base_path = f"/Volumes/ecommerce/source_data/archive/{archive_date}/date/"
print("archive_base_path : ",archive_base_path)

try:
    # Write to Bronze Delta Table
    (
        df_raw.write
              .format("delta")
              .mode("overwrite")
              .option("mergeSchema", "true")
              .saveAsTable(f"{catalog_name}.bronze.brz_calendar")
    )

    # Create archive folder
    dbutils.fs.mkdirs(archive_base_path)

    # Get all processed source files
    source_files = (
        df_raw.select("_source_file")
              .distinct()
              .collect()
    )
    print("source_files : ",source_files)
    # Archive each file
    for row in source_files:
        source_file = row["_source_file"]
        file_name = os.path.basename(source_file)

        destination_file = f"{archive_base_path}{file_name}"
        
        dbutils.fs.mv(source_file, destination_file)

        print(f"Archived: {source_file} -> {destination_file}")

    print("Delta load and file archival completed successfully.")

except Exception as e:
    print(f"Pipeline failed: {str(e)}")
    raise
            

archive_base_path :  /Volumes/ecommerce/source_data/archive/2026-06-20/date/
source_files :  [Row(_source_file='dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv')]
Archived: dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv -> /Volumes/ecommerce/source_data/archive/2026-06-20/date/date.csv
Delta load and file archival completed successfully.
